In [20]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [21]:
import h5py
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


In [22]:
class PatchCamHDF5(Dataset):
    def __init__(self, x_path, y_path=None, transform=None):
        self.x_file = h5py.File(x_path, 'r')
        self.x = self.x_file['x']
        
        self.y = None
        if y_path is not None:
            self.y_file = h5py.File(y_path, 'r')
            self.y = self.y_file['y']
        
        self.transform = transform

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        img = self.x[idx]              # (96,96,3), uint8
        img = img.astype('float32') / 255.0

        if self.transform:
            img = self.transform(img)

        if self.y is not None:
            label = int(self.y[idx])
            return img, label
        else:
            return img


In [23]:
transform = transforms.Compose([
    transforms.ToTensor(),                 # (H,W,C) → (C,H,W)
    transforms.Normalize([0.5]*3, [0.5]*3) # [-1,1]
])


In [28]:
import os

print(os.listdir("/kaggle/input"))


['patchcam-dataset']


In [29]:
DATA_ROOT = "/kaggle/input/patchcam-dataset"

train_x = f"{DATA_ROOT}/camelyonpatch_level_2_split_train_x.h5"
train_y = f"{DATA_ROOT}/camelyonpatch_level_2_split_train_y.h5"

dataset = PatchCamHDF5(
    x_path=train_x,
    y_path=train_y,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,      # Kaggle safe
    pin_memory=True,
    drop_last=True
)

print("Total images:", len(dataset))


Total images: 262144


In [32]:
imgs, labels = next(iter(dataloader))
print(imgs.shape)    # [B,3,96,96]
print(labels.unique())
print(imgs.min(), imgs.max())


/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


torch.Size([64, 3, 96, 96])
tensor([0, 1])
tensor(-1.) tensor(1.)


In [33]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 6, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


In [34]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf*8, 1, 6, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)


In [35]:
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()

optG = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
optD = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))


In [36]:
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

for epoch in range(epochs):
    for i, (real, _) in enumerate(dataloader):
        real = real.to(device)
        bsz = real.size(0)

        # ---- Train Discriminator ----
        D.zero_grad()
        label_real = torch.ones(bsz, device=device)
        label_fake = torch.zeros(bsz, device=device)

        out_real = D(real)
        lossD_real = criterion(out_real, label_real)

        noise = torch.randn(bsz, nz, 1, 1, device=device)
        fake = G(noise)
        out_fake = D(fake.detach())
        lossD_fake = criterion(out_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        # ---- Train Generator ----
        G.zero_grad()
        out_fake = D(fake)
        lossG = criterion(out_fake, label_real)
        lossG.backward()
        optG.step()

        if i % 200 == 0:
            print(f"[Epoch {epoch}/{epochs}] "
                  f"[Batch {i}/{len(dataloader)}] "
                  f"D: {lossD.item():.3f} | G: {lossG.item():.3f}")


/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


[Epoch 0/25] [Batch 0/4096] D: 1.440 | G: 3.505
[Epoch 0/25] [Batch 200/4096] D: 0.807 | G: 1.485
[Epoch 0/25] [Batch 400/4096] D: 1.694 | G: 3.857
[Epoch 0/25] [Batch 600/4096] D: 1.185 | G: 5.997
[Epoch 0/25] [Batch 800/4096] D: 0.456 | G: 8.266
[Epoch 0/25] [Batch 1000/4096] D: 0.795 | G: 2.613
[Epoch 0/25] [Batch 1200/4096] D: 0.445 | G: 9.170
[Epoch 0/25] [Batch 1400/4096] D: 0.562 | G: 10.438
[Epoch 0/25] [Batch 1600/4096] D: 0.068 | G: 3.791
[Epoch 0/25] [Batch 1800/4096] D: 0.571 | G: 4.970
[Epoch 0/25] [Batch 2000/4096] D: 1.652 | G: 7.749
[Epoch 0/25] [Batch 2200/4096] D: 0.391 | G: 7.849
[Epoch 0/25] [Batch 2400/4096] D: 0.594 | G: 8.370
[Epoch 0/25] [Batch 2600/4096] D: 0.838 | G: 5.485
[Epoch 0/25] [Batch 2800/4096] D: 0.521 | G: 9.057
[Epoch 0/25] [Batch 3000/4096] D: 0.057 | G: 4.630
[Epoch 0/25] [Batch 3200/4096] D: 0.092 | G: 4.871
[Epoch 0/25] [Batch 3400/4096] D: 0.184 | G: 5.095
[Epoch 0/25] [Batch 3600/4096] D: 0.844 | G: 3.750
[Epoch 0/25] [Batch 3800/4096] D: 0.7

/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


[Epoch 1/25] [Batch 0/4096] D: 0.273 | G: 5.333
[Epoch 1/25] [Batch 200/4096] D: 0.146 | G: 6.430
[Epoch 1/25] [Batch 400/4096] D: 0.308 | G: 8.342
[Epoch 1/25] [Batch 600/4096] D: 0.248 | G: 7.062
[Epoch 1/25] [Batch 800/4096] D: 0.201 | G: 6.219
[Epoch 1/25] [Batch 1000/4096] D: 0.244 | G: 5.404
[Epoch 1/25] [Batch 1200/4096] D: 0.037 | G: 8.182
[Epoch 1/25] [Batch 1400/4096] D: 0.178 | G: 7.633
[Epoch 1/25] [Batch 1600/4096] D: 0.148 | G: 8.036
[Epoch 1/25] [Batch 1800/4096] D: 0.007 | G: 4.836
[Epoch 1/25] [Batch 2000/4096] D: 0.194 | G: 8.388
[Epoch 1/25] [Batch 2200/4096] D: 0.101 | G: 6.114
[Epoch 1/25] [Batch 2400/4096] D: 0.102 | G: 5.687
[Epoch 1/25] [Batch 2600/4096] D: 0.055 | G: 7.340
[Epoch 1/25] [Batch 2800/4096] D: 0.073 | G: 5.107
[Epoch 1/25] [Batch 3000/4096] D: 0.003 | G: 11.985
[Epoch 1/25] [Batch 3200/4096] D: 0.082 | G: 7.237
[Epoch 1/25] [Batch 3400/4096] D: 0.033 | G: 25.316
[Epoch 1/25] [Batch 3600/4096] D: 0.019 | G: 11.145
[Epoch 1/25] [Batch 3800/4096] D: 0

/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


...

[Epoch 22/25] [Batch 0/4096] D: 0.091 | G: 5.249
[Epoch 22/25] [Batch 200/4096] D: 0.156 | G: 7.237
[Epoch 22/25] [Batch 400/4096] D: 0.199 | G: 10.985
[Epoch 22/25] [Batch 600/4096] D: 0.013 | G: 6.075
[Epoch 22/25] [Batch 800/4096] D: 0.135 | G: 3.904
[Epoch 22/25] [Batch 1000/4096] D: 0.023 | G: 5.551
[Epoch 22/25] [Batch 1200/4096] D: 0.064 | G: 7.077
[Epoch 22/25] [Batch 1400/4096] D: 0.014 | G: 6.268
[Epoch 22/25] [Batch 1600/4096] D: 0.008 | G: 6.847
[Epoch 22/25] [Batch 1800/4096] D: 0.059 | G: 4.055
[Epoch 22/25] [Batch 2000/4096] D: 0.029 | G: 4.793
[Epoch 22/25] [Batch 2200/4096] D: 0.029 | G: 5.731
[Epoch 22/25] [Batch 2400/4096] D: 0.089 | G: 3.788
[Epoch 22/25] [Batch 2600/4096] D: 0.023 | G: 7.185
[Epoch 22/25] [Batch 2800/4096] D: 0.205 | G: 2.471
[Epoch 22/25] [Batch 3000/4096] D: 0.110 | G: 6.268
[Epoch 22/25] [Batch 3200/4096] D: 0.026 | G: 6.186
[Epoch 22/25] [Batch 3400/4096] D: 0.042 | G: 4.802
[Epoch 22/25] [Batch 3600/4096] D: 0.017 | G: 5.750
[Epoch 22/25] [Bat

/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


[Epoch 23/25] [Batch 0/4096] D: 0.028 | G: 6.590
[Epoch 23/25] [Batch 200/4096] D: 0.197 | G: 8.262
[Epoch 23/25] [Batch 400/4096] D: 0.069 | G: 6.713
[Epoch 23/25] [Batch 600/4096] D: 0.035 | G: 5.766
[Epoch 23/25] [Batch 800/4096] D: 0.017 | G: 5.908
[Epoch 23/25] [Batch 1000/4096] D: 0.038 | G: 5.985
[Epoch 23/25] [Batch 1200/4096] D: 0.005 | G: 8.184
[Epoch 23/25] [Batch 1400/4096] D: 0.008 | G: 7.467
[Epoch 23/25] [Batch 1600/4096] D: 0.061 | G: 6.352
[Epoch 23/25] [Batch 1800/4096] D: 0.021 | G: 6.444
[Epoch 23/25] [Batch 2000/4096] D: 0.021 | G: 7.149
[Epoch 23/25] [Batch 2200/4096] D: 0.012 | G: 6.102
[Epoch 23/25] [Batch 2400/4096] D: 0.030 | G: 5.770
[Epoch 23/25] [Batch 2600/4096] D: 0.136 | G: 6.974
[Epoch 23/25] [Batch 2800/4096] D: 0.043 | G: 9.039
[Epoch 23/25] [Batch 3000/4096] D: 0.040 | G: 6.222
[Epoch 23/25] [Batch 3200/4096] D: 0.100 | G: 4.704
[Epoch 23/25] [Batch 3400/4096] D: 0.062 | G: 4.458
[Epoch 23/25] [Batch 3600/4096] D: 0.021 | G: 5.486
[Epoch 23/25] [Batc

/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


[Epoch 24/25] [Batch 0/4096] D: 0.039 | G: 5.642
[Epoch 24/25] [Batch 200/4096] D: 0.013 | G: 6.314
[Epoch 24/25] [Batch 400/4096] D: 0.009 | G: 7.671
[Epoch 24/25] [Batch 600/4096] D: 0.084 | G: 7.275
[Epoch 24/25] [Batch 800/4096] D: 0.037 | G: 6.063
[Epoch 24/25] [Batch 1000/4096] D: 0.030 | G: 6.285
[Epoch 24/25] [Batch 1200/4096] D: 0.038 | G: 7.578
[Epoch 24/25] [Batch 1400/4096] D: 0.008 | G: 6.139
[Epoch 24/25] [Batch 1600/4096] D: 0.041 | G: 5.198
[Epoch 24/25] [Batch 1800/4096] D: 0.016 | G: 6.857
[Epoch 24/25] [Batch 2000/4096] D: 0.059 | G: 5.738
[Epoch 24/25] [Batch 2200/4096] D: 0.059 | G: 5.096
[Epoch 24/25] [Batch 2400/4096] D: 0.086 | G: 6.356
[Epoch 24/25] [Batch 2600/4096] D: 0.431 | G: 8.831
[Epoch 24/25] [Batch 2800/4096] D: 0.047 | G: 5.477
[Epoch 24/25] [Batch 3000/4096] D: 0.024 | G: 5.741
[Epoch 24/25] [Batch 3200/4096] D: 0.031 | G: 5.340
[Epoch 24/25] [Batch 3400/4096] D: 0.052 | G: 5.388
[Epoch 24/25] [Batch 3600/4096] D: 0.003 | G: 6.878
[Epoch 24/25] [Batc

In [40]:
!pip install torch-fidelity pytorch-msssim


In [41]:
from torch_fidelity import calculate_metrics
from pytorch_msssim import ssim

def collect_images(G, loader, n):
    real, fake = [], []
    G.eval()
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            z = torch.randn(imgs.size(0), nz, 1, 1, device=device)
            gen = G(z)
            real.append(imgs)
            fake.append(gen)
            if len(real)*imgs.size(0) >= n:
                break
    return torch.cat(real)[:n], torch.cat(fake)[:n]

real_imgs, fake_imgs = collect_images(G, dataloader, num_metric_samples)


/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])
/tmp/ipykernel_55/2533069667.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(self.y[idx])


In [48]:
from torch.utils.data import Dataset

class TensorImageDataset(Dataset):
    def __init__(self, images):
        self.images = images  # MUST be CPU tensor

    def __len__(self):
        return self.images.size(0)

    def __getitem__(self, idx):
        return self.images[idx]


In [47]:
real_fid = real_fid.cpu()
fake_fid = fake_fid.cpu()


In [49]:
real_ds = TensorImageDataset(real_fid)
fake_ds = TensorImageDataset(fake_fid)

metrics = calculate_metrics(
    input1=real_ds,
    input2=fake_ds,
    fid=True,
    isc=False,
    kid=False,
    cuda=False,        # ❗ force CPU
    num_workers=0,     # ❗ ABSOLUTELY REQUIRED on Kaggle
    batch_size=32,
    verbose=False
)

print("FID:", metrics["frechet_inception_distance"])


FID: 70.81127560993548


In [50]:
ssim_val = ssim(
    (real_imgs + 1)/2,
    (fake_imgs + 1)/2,
    data_range=1.0,
    size_average=True
)

print("SSIM:", ssim_val.item())


SSIM: 0.05798017978668213


In [51]:
def compute_mmd(x, y, sigma=1.0):
    x = x.view(x.size(0), -1).cpu().numpy()
    y = y.view(y.size(0), -1).cpu().numpy()

    xx = np.exp(-cdist(x, x) / (2*sigma**2))
    yy = np.exp(-cdist(y, y) / (2*sigma**2))
    xy = np.exp(-cdist(x, y) / (2*sigma**2))

    return xx.mean() + yy.mean() - 2*xy.mean()

print("MMD:", compute_mmd(real_imgs, fake_imgs))


MMD: 0.0020071277884045214


In [55]:
# --- Final consolidated output ---
print("📊 DCGAN Evaluation Metrics")
print("---------------------------")
print(f"FID  : {metrics["frechet_inception_distance"]:.4f}")
print(f"SSIM : {ssim_val.item():.4f}")
print(f"MMD  : {compute_mmd(real_imgs, fake_imgs):.6f}")

📊 DCGAN Evaluation Metrics
---------------------------
FID  : 70.8113
SSIM : 0.0580
MMD  : 0.002007


In [52]:
import os
import torch

SAVE_DIR = "/kaggle/working/models"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "generator_state_dict": G.state_dict(),
    "nz": nz,
    "image_size": image_size
}, f"{SAVE_DIR}/dcgan_generator_patchcam.pth")

print("✅ Generator saved at:", f"{SAVE_DIR}/dcgan_generator_patchcam.pth")


✅ Generator saved at: /kaggle/working/models/dcgan_generator_patchcam.pth
